# Name That LLM: GPU-Accelerated Transformer Fine-Tuning & Pipeline
### Google Colab & Kaggle Notebook (T4, P100, L4, A100)

This notebook trains a deep transformer classifier (DeBERTa-v3 / RoBERTa / ModernBERT) across the 18 frontier LLM targets with:
1. **Strict Prompt-Grouped Folds** (`prompt_id` grouping to prevent topic leakage)
2. **Class-Balanced CrossEntropyLoss** (directly optimizing Balanced Accuracy)
3. **Mixed Precision (`fp16`/`bf16`)** & Gradient Clipping
4. **Out-Of-Fold Validation & Submission Generation** (`submission.parquet`)

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dependencies Installation
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow fastparquet

In [ ]:
# 3. Kaggle API Setup & Data Loading
import os, glob
import numpy as np, pandas as pd

OFFICIAL_18_CLASSES = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
LABEL_TO_ID = {name: i for i, name in enumerate(OFFICIAL_18_CLASSES)}
ID_TO_LABEL = {i: name for i, name in enumerate(OFFICIAL_18_CLASSES)}

def find_file(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

# Locate training data (Kaggle or local)
train_path = find_file('/kaggle/input/**/train.csv') or find_file('**/passages_clean.csv') or 'data/ref_dataset/passages_clean.csv'
print(f"Loading data from: {train_path}")
df = pd.read_csv(train_path)
if 'model' not in df.columns and 'label' in df.columns:
    df['model'] = df['label']
if 'text' not in df.columns and 'passage' in df.columns:
    df['text'] = df['passage']

df = df[df['model'].isin(OFFICIAL_18_CLASSES)].dropna(subset=['text', 'prompt_id']).reset_index(drop=True)
df['label_id'] = df['model'].map(LABEL_TO_ID)
print(f"Samples: {len(df):,}, Prompts: {df['prompt_id'].nunique()}, Classes: {df['model'].nunique()}")

In [ ]:
# 4. Strict Stratified Group Folds on prompt_id
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(sgkf.split(df, y=df['label_id'], groups=df['prompt_id'])):
    df.loc[val_idx, 'fold'] = fold

# Leakage assertion
for f in range(5):
    tr_p = set(df[df['fold'] != f]['prompt_id'])
    val_p = set(df[df['fold'] == f]['prompt_id'])
    assert len(tr_p.intersection(val_p)) == 0, f"Leakage in fold {f}!"
print("GroupKFold verified: 0 prompt overlap across all 5 folds.")

In [ ]:
# 5. Transformer Model Training Setup
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import balanced_accuracy_score

MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 256
BATCH_SIZE = 8
LR = 2.5e-5
EPOCHS = 6
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NTLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0)}
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f'Ready to train {MODEL_NAME} on {DEVICE}!')


In [ ]:
# 6. Class-Weighted Balanced Training with Checkpointing
train_sub = df[df['fold'] != 0].reset_index(drop=True)
val_sub = df[df['fold'] == 0].reset_index(drop=True)

# Compute inverse frequency class weights to prevent majority collapse
class_counts = train_sub['label_id'].value_counts().sort_index().values
weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32).to(DEVICE)
weights = (weights / weights.sum()) * 18.0
criterion = nn.CrossEntropyLoss(weight=weights)

train_loader = DataLoader(NTLDataset(train_sub['text'], train_sub['label_id'], tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NTLDataset(val_sub['text'], val_sub['label_id'], tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=18).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, eps=1e-6, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(len(train_loader)*0.1), num_training_steps=len(train_loader)*EPOCHS)

best_acc = 0.0
print(f'Starting class-weighted training for {EPOCHS} epochs on {DEVICE}...')
for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        ids, mask, y = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
        out = model(ids, attention_mask=mask)
        loss = criterion(out.logits.float(), y)
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item()
    
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids, mask, y = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
            out = model(ids, attention_mask=mask)
            preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
            targets.extend(y.cpu().numpy())
    bal_acc = balanced_accuracy_score(targets, preds)
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {tr_loss/len(train_loader):.4f} | Val Balanced Acc: {bal_acc:.4f}')
    if bal_acc > best_acc:
        best_acc = bal_acc
        model.save_pretrained('./saved_model')
        tokenizer.save_pretrained('./saved_model')
        print(f'New Best Model saved with Balanced Acc: {best_acc:.4f}')

print(f'Training complete! Peak Balanced Acc: {best_acc:.4f}')


In [ ]:
# 7. Generate Competition Compliant submission.parquet
test_path = find_file('/kaggle/input/**/test.csv') or 'data/test.csv'
test_df = pd.read_csv(test_path)
test_loader = DataLoader(NTLDataset(test_df['text'], None, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        ids, mask = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)
        out = model(ids, attention_mask=mask)
        test_preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())

sub = pd.DataFrame({'id': test_df['id'], 'model': [ID_TO_LABEL[i] for i in test_preds]})
sub.to_parquet('submission.parquet', index=False)
print(f"Saved submission.parquet with {len(sub)} rows!")
print(sub.head())
print(sub['model'].value_counts())
